# Coffee Quality: EDA and Modelling

A worked example of the machine learning workflow. We use the Arabica coffee quality dataset to predict each coffee's overall cupping score from its measured attributes, then check how well the model does. Treat this notebook as a reference for the structure of your own project.

## The Machine Learning Workflow

### Define the Business Goal

The stakeholder is a **green coffee buyer at a coffee importer**. Each season they review many Arabica lots and want to spend their time and budget on the most promising ones.

Estimate a coffee's overall **cupping score** from attributes known *before* a full tasting, so the buyer can shortlist which lots to taste, support price negotiations, and avoid spending cupping sessions on low-quality coffees.

The score depends on many attributes at once, so a model gives a faster and more consistent estimate than a manual rule of thumb.

The estimate should be close enough to be useful in practice, for example within a couple of points on the 0 to 100 scale.

In data science terms this is a **regression** problem: the target `total_cup_points` is a continuous number, and we measure the error with the **root mean squared error (RMSE)**, which is in the same units as the score (points).

### Get the Data

We load the cleaned Arabica dataset. It holds both the coffee attributes and the overall cupping score (`Total.Cup.Points`), which we will use as the target.

In [ ]:
import pandas as pd

# arabica coffee data (cleaned)
URL = "https://raw.githubusercontent.com/jldbc/coffee-quality-database/master/data/arabica_data_cleaned.csv"
coffee = pd.read_csv(URL)

Let us preview the data to see what we are working with.

In [ ]:
coffee.head()

### First Look at the Data

A quick structural check shows the column types and where values are missing.

In [ ]:
coffee.info()

The dataset has 1311 Arabica coffees described by 44 columns: a mix of text (origin, farm, processing method) and numbers (the cupping scores, altitude, defect counts). Several columns have missing values, and the altitude column also contains some impossible entries. We clean all of this next.

## Clean the Data

These steps are all **leakage-safe**, so we do them before the split: tidy the column names, drop columns and rows we cannot use, fix obvious data-entry errors, and remove features that would give away the answer. The only step we leave for after the split is imputation, because its fill values are learned from the data.

In [ ]:
# Clean the column names: lowercase, with underscores instead of dots and spaces
coffee.columns = (
    coffee.columns.str.strip()
    .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
    .str.strip("_")
    .str.lower()
)
coffee.columns.tolist()

The messy names (for example `Category.One.Defects` and `Total.Cup.Points`) become tidy snake_case. This is pure renaming, so it is always safe.

In [ ]:
# Drop columns that are missing more than half of their values
missing_fraction = coffee.isna().mean().sort_values(ascending=False)
high_missing = missing_fraction[missing_fraction > 0.5].index
print("Dropping columns missing more than 50% of values:", list(high_missing))
coffee = coffee.drop(columns=high_missing)

Only `lot_number` is missing more than half of its values, so we drop it.

In [ ]:
# Fix altitude data-entry errors: an altitude of 10000 metres or more is impossible
# (no coffee grows that high), so it is almost certainly an order-of-magnitude typo.
# Rescale any such value down by 10 until it reaches a believable range.
altitude_cols = ["altitude_low_meters", "altitude_high_meters", "altitude_mean_meters"]
print("Altitude values of 10000+ before fix:", int((coffee[altitude_cols] >= 10000).sum().sum()))
for col in altitude_cols:
    while (coffee[col] >= 10000).any():
        coffee[col] = coffee[col].where(coffee[col] < 10000, coffee[col] / 10)
print("Highest altitude after fix:", round(coffee["altitude_mean_meters"].max(), 1))

The altitude columns hold impossible values such as 190164 metres. These look like order-of-magnitude typos, so we rescale any altitude of 10000 or more down by factors of 10 until it is believable. This is a fixed, per-row rule, so it is leakage-safe. It rescales 12 values across the three altitude columns, and the highest altitude drops to about 4287 metres.

In [ ]:
# One coffee has an overall score of 0, which is impossible. Drop that row.
print("Rows with a total score of 0:", int((coffee["total_cup_points"] == 0).sum()))
coffee = coffee[coffee["total_cup_points"] > 0]

Every other coffee scores between about 59 and 91, so a score of 0 is clearly a data-entry error. We drop that single row, leaving 1310 coffees.

In [ ]:
# Domain knowledge: the overall score is the SUM of the component cupping scores.
# Confirm it, then treat those components as leakage.
component_scores = [
    "aroma", "flavor", "aftertaste", "acidity", "body",
    "balance", "uniformity", "clean_cup", "sweetness", "cupper_points",
]
sum_corr = coffee[component_scores].sum(axis=1).corr(coffee["total_cup_points"])
print("Correlation of the component sum with the total:", round(sum_corr, 3))

The component scores add up to the total (correlation 1.00). That makes them **target leakage**: feeding them to the model would just hand it the answer broken into ten pieces. They are also unrealistic for our goal, because the buyer wants an estimate *before* tasting, when these cupping scores do not exist yet.

So we drop the component scores now and predict the quality from genuinely independent attributes instead.

In [ ]:
# The target is the overall cupping score
y = coffee["total_cup_points"]

# Features: numeric columns, minus the row index, the target, and the leaky component scores
X = coffee.select_dtypes("number").drop(
    columns=["unnamed_0", "total_cup_points"] + component_scores
)
X.columns.tolist()

This leaves eight independent attributes: the number of bags, moisture, the two defect counts, quakers, and the three altitude columns. These are things a buyer could know about a lot without cupping it.

## Train-Test Split

We hold out part of the data so we can judge the model on coffees it has never seen. From here on, anything we *fit* (such as imputation values) is learned on the training set only.

We use a single train-test split (80% train, 20% test). A larger project would also keep a validation set, or use cross validation, to tune the model without touching the test set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

The 1310 coffees are split into 1048 for training and 262 for testing.

## Explore the Training Data

Now we explore the **training set** to understand the features before fitting anything.

In [ ]:
X_train.info()

There are eight numeric features. Most are complete, but `quakers` has one missing value and the three altitude columns are missing for 187 of the 1048 training coffees. We will impute those gaps after the split.

In [ ]:
# How does each remaining feature correlate with the target?
X_train.corrwith(y_train).sort_values(ascending=False)

None of the independent attributes is strongly related to the score. The clearest signal is `category_two_defects` (about -0.28: more defects, lower score), followed by moisture and altitude, all weak.

## Feature Engineering

We fill every missing value (`quakers` and the three altitude columns) with the **training** mean of its column, then reuse those values on the test set.

In [ ]:
# Learn the fill values on the training set only
train_means = X_train.mean()
X_train = X_train.fillna(train_means)
train_means

`train_means` holds one value per column. We reuse this exact set of values on the test set at evaluation time, which is what keeps the evaluation honest.

## Baseline Model

Before training a model, we set a **baseline**: the simplest rule that makes a prediction.

For a regression like ours, a standard heuristic is to ignore the features and always predict the **average training score**. If our model cannot beat that, it has learned nothing useful.

In [ ]:
# Heuristic baseline: ignore the features, always predict the mean training score
from sklearn.metrics import root_mean_squared_error

baseline_prediction = y_train.mean()
baseline_rmse = root_mean_squared_error(y_test, [baseline_prediction] * len(y_test))

print(f"Baseline always predicts: {baseline_prediction:.2f}")
print(f"Baseline test RMSE: {baseline_rmse:.2f}")

The baseline always predicts about 82.2 (the mean training score) and reaches a test RMSE of about 2.66, so on average it misses the true score by roughly 2 to 3 points. This is the number our model has to beat.

## Train the Model

We train a linear regression model: fast, easy to read, and a sensible step up from the baseline. First we set the test set aside and save a copy, then we fit the model and check the error on the training data.

In [ ]:
# Set the test set aside and save a copy. We do not touch it until the final evaluation.
X_test.to_csv("data/X_test.csv")
y_test.to_csv("data/y_test.csv")

In [ ]:
from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X_train, y_train)

In [ ]:
y_train_pred = reg.predict(X_train)
rmse = root_mean_squared_error(y_train, y_train_pred)
print(rmse)

The training RMSE is about 2.54 points, close to the baseline. The model is not memorising the data, which is unsurprising given how weakly the features relate to the score.

## Evaluate the Model

We apply the same fitted step to the test set: imputation with the **training** means (never recomputed from the test data). The cleaning and the altitude fix already happened before the split, so the test set carries them too. Then we score the model on these unseen coffees.

In [ ]:
# Apply the fitted step (imputation) to the test set, using the training means
X_test = X_test.fillna(train_means)

In [ ]:
y_test_pred = reg.predict(X_test)
rmse = root_mean_squared_error(y_test, y_test_pred)
print(rmse)

The test RMSE is about 2.46, just below the baseline's 2.66. The model is only a little better than always guessing the average: the attributes a buyer can know before tasting (altitude, defects, moisture, bag count) carry only weak signal about the final cupping score.

The natural next steps in the workflow,  are trying richer features or models, selecting the best one on validation data or cross-validation, hyperparameter tuning, doing error analysis, reporting a final test score, then deploying and monitoring the model.

In [ ]:
# save the best model to a file for later use
import joblib
joblib.dump(reg, "models/linear_regression_model.joblib")

In [ ]:
# use the model to predict the score of a new coffee with these attributes:
 
new_coffee = pd.DataFrame({
    
    "number_of_bags" :[1] ,
    "moisture": [10.5] , 
    "category_one_defects": [0],
    "quakers": [0],
    "category_two_defects": [1], 
    "altitude_low_meters": [1200], 
    "altitude_high_meters": [1500],
    "altitude_mean_meters": [1350]
})
new_coffee = new_coffee.fillna(train_means)  # apply the same imputation as training data

In [ ]:
X_train.columns

In [ ]:
# load the model from the file and use it to predict the score of the new coffee
loaded_model = joblib.load("models/linear_regression_model.joblib")
predicted_score = loaded_model.predict(new_coffee)[0]
print(f"Predicted cupping score for the new coffee: {predicted_score:.2f}")

## References & Further Reading

- [**LinearRegression (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html): the model trained in this notebook.
- [**train_test_split (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html): how the held-out test set is created.
- [**root_mean_squared_error (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html): the metric used to evaluate the model and the baseline.
- [**Common pitfalls and recommended practices (scikit-learn)**](https://scikit-learn.org/stable/common_pitfalls.html): how data leakage happens and how to avoid it.
- [**Coffee Quality Database**](https://github.com/jldbc/coffee-quality-database): the source of the dataset used here.